In [ ]:
import numpy as np
import math
import random
import time
class Kernel_SGD():
    def __init__(self,class_num,d,s,loss,kernel,sigma):
        self.class_num = class_num
        self.d = d
        self.Alpha_last = np.zeros((1,self.class_num))
        self.Alpha_ave = np.zeros((1,self.class_num))
        self.X = [np.zeros(self.d)]
        self.s1 = s
        self.cc_B4 = -math.pi**4/48
        self.cc_B2 = math.pi**2/4
        self.loss = loss
        self.kernel = kernel
        self.sigma = sigma

    def kernel_com(self,x1,x2):
        t = x1-x2
        if self.kernel == 'matern':
            tau = np.sqrt(np.dot(t,t))
            value = (1+np.sqrt(5)*(tau)+5*(tau**2)/3)*np.exp(-np.sqrt(5)*tau)
        elif self.kernel == 'gauss':
            tau = np.dot(t,t)
            value = np.exp(-tau/2)
        elif self.kernel == 'B2':
            tau = np.sqrt(np.dot(t,t))
            fraction_xy = tau-int(tau)
            value = ((fraction_xy-1)*fraction_xy+1/6)*self.cc_B2
        elif self.kernel == 'B4':
            tau = np.sqrt(np.dot(t,t))
            fraction_xy = tau-int(tau)
            value = (((fraction_xy-2)*fraction_xy+1)*fraction_xy**2-1/30)*self.cc_B4
        return value

    def computation_f_last(self,x):
        S = np.zeros(self.class_num)
        if len(self.Alpha_last) > 0:
            A = np.array([ self.kernel_com(a,x) for a in self.X])
            S = np.dot(A,self.Alpha_last)
        return S

    def computation_f_ave(self,x):
        S = np.zeros(self.class_num)
        if len(self.Alpha_ave) > 0:
            A = np.array([ self.kernel_com(a,x) for a in self.X])
            S = np.dot(A,self.Alpha_ave)
        return S

    def derivative_loss(self,X,Y):
        if self.loss == 'least square':
            derivative = (self.computation_f_last(X)-Y)
        elif self.loss == 'logistic':
            term = np.exp(Y*self.computation_f_last(X))
            derivative = - Y / (1+term)
        elif self.loss == 'Cauchy':
            term = self.computation_f_last(X)-Y
            derivative = term/(self.sigma**2+term**2/2)
        elif self.loss == 'Huber':
            term = self.computation_f_last(X)-Y
            derivative = term/np.sqrt(term**2+1)
        elif self.loss == 'Welsch':
            term = self.computation_f_last(X)-Y
            derivative = np.exp(-term**2/(2*self.sigma**2))*(term/self.sigma**2)
        return derivative
            
    def updata(self,x,y,epoch):
        self.gamma = self.gamma0*((epoch)**(-self.t))
        coe = -self.gamma*self.derivative_loss(x,y)
        self.Alpha_last = np.append(self.Alpha_last,coe.reshape(1,self.class_num),axis =0)
        self.Alpha_ave = np.append(self.Alpha_ave, np.zeros((1,self.class_num)), axis = 0)
        self.Alpha_ave = (epoch*self.Alpha_ave +self.Alpha_last)/(epoch+1)
        self.X.append(x)

    def grho_cal(self,theta):
        x = theta
        return (x**2-x+1/6)/5 #(t**4-2*(t**3)+t**2-1/30)/8
    
    def construct_data(self):
        theta = random.random()
        Y = self.grho_cal(theta)
        Z = Y + random.uniform(-0.2,0.2)
        return [theta,Z,Y]

    def test(self):
        S = 0
        for i in range(2000):
            data = self.construct_data()
            y = self.computation_f_last(data[0])
            S += np.dot(y-data[2],y-data[2])
        return S/2000
    
    def algorthim(self,t,gamma0,Epoch,NN,Uni):
        self.Error = []
        self.Time = []
        start = time.perf_counter()
        self.t = t
        self.gamma0 = gamma0
        for epoch in range(Epoch):
            Z = self.construct_data()
            self.updata(Z[0],Z[1],epoch+1)
            if epoch in NN:
                if Uni == "time":
                    end = time.perf_counter()
                    runTime = end - start
                    self.Time.append(runTime)
                    #print(epoch,'time',runTime)
                elif Uni == 'error':
                    err = self.test()
                    self.Error.append(err)
                    #print(epoch,'err',err)
        return self.Error
            
        
        

In [ ]:
d = 1
N = [int(10**(i/5))for i in range(1,20)]
s = 1.
r=3/4
gamma0 = 1.6
t = 2*r/(2*r+1)
theta = 1/(2*s*(2*r+1))
Sum_epoch = int(N[-1]+1)
sigma = 1.
T = 10
Loss = ['Cauchy', 'Huber', 'Welsch']
Kernel = ['matern','gauss','B2']
for loss in Loss:
    for kernel in Kernel:
        err = []
        err_ave = []
        for i in range(T):
            kernel_SGD = Kernel_SGD(1,d,10**(-8*i),loss,kernel,sigma)
            kernel_SGD.algorthim(t,gamma0,Sum_epoch,N,'error')
            err.append(kernel_SGD.Error[:])
        for i in range(len(err[-1])):
            S = 0
            for j in range(T):
                S += err[j][i]
            err_ave.append(S/T)
        if loss == 'least square':
            value = 'least'
        else:
            value = loss
        print('err_kernel_SGD_'+kernel+'_'+value+'_B2=',err_ave)
    print(' ')

In [ ]:
d = 1
N = [int(10**(i/5))for i in range(1,20)]
s = 1.
r=3/4
gamma0 =1.6
t = 2*r/(2*r+1)
theta = 1/(2*s*(2*r+1))
Sum_epoch = int(N[-1]+1)
T = 5
Loss = ['Cauchy', 'Huber', 'Welsch']
Kernel = ['matern','gauss','B2']
for loss in Loss:
    for kernel in Kernel:
        Time = []
        Time_ave = []
        for i in range(T):
            kernel_SGD = Kernel_SGD(1,d,10**(-8*i),loss,kernel,sigma)
            kernel_SGD.algorthim(t,gamma0,Sum_epoch,N,'time')
            Time.append(kernel_SGD.Time[:])
        for i in range(len(Time[-1])):
            S = 0
            for j in range(T):
                S += Time[j][i]
            Time_ave.append(S/T)
        if loss == 'least square':
            value = 'least'
        else:
            value = loss
        print('time_'+kernel+'_'+value+'_B2=',Time_ave)
    print(' ')